# 02_agrupar_por_lat_lon

Este notebook carga los datos limpios generados en `01_read_filter_data.ipynb`, prepara las características de latitud/longitud y aplica K-Means para encontrar zonas calientes de biodiversidad.

In [1]:
import os

jdk_path = os.path.expanduser('~/.jdk17')
if os.path.isdir(jdk_path):
    os.environ['JAVA_HOME'] = jdk_path
    os.environ['PATH'] = os.path.join(jdk_path, 'bin') + os.pathsep + os.environ.get('PATH', '')
    print('JAVA_HOME seteado a', jdk_path)
else:
    print('No se encontró JDK en', jdk_path)

JAVA_HOME seteado a /home/codespace/.jdk17


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import os

base_dir = os.getcwd()
candidate = os.path.join(base_dir, 'biodiversidad_limpa.parquet')
fallback = os.path.join(base_dir, 'Notebook', 'biodiversidad_limpa.parquet')
if os.path.isdir(candidate):
    data_path = candidate
elif os.path.isdir(fallback):
    data_path = fallback
else:
    raise FileNotFoundError(f'No se encontró el parquet limpio en ninguna ruta: {candidate} o {fallback}')

print('Leyendo datos limpios desde', data_path)

spark = SparkSession.builder \
    .appName('BiodiversidadColombiaAgrupacion') \
    .master('local[*]') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

df = spark.read.parquet(data_path)
print('Esquema de los datos limpios:')
df.printSchema()
print('Muestra de datos:')
df.show(10, truncate=False)

Leyendo datos limpios desde /workspaces/bigData-Class/Notebook/biodiversidad_limpa.parquet


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/29 05:24:27 WARN Utils: Your hostname, codespaces-e48955, resolves to a loopback address: 127.0.0.1; using 10.0.0.90 instead (on interface eth0)
26/05/29 05:24:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/29 05:24:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/29 05:24:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Esquema de los datos limpios:
root
 |-- species: string (nullable = true)
 |-- decimalLatitude: double (nullable = true)
 |-- decimalLongitude: double (nullable = true)

Muestra de datos:


+-----------------------+---------------+----------------+
|species                |decimalLatitude|decimalLongitude|
+-----------------------+---------------+----------------+
|Guazuma ulmifolia      |10.25733       |-73.365369      |
|Handroanthus billbergii|10.1007        |-73.630665      |
|Handroanthus billbergii|10.26731       |-73.349081      |
|Handroanthus billbergii|10.09897       |-73.634235      |
|Senegalia polyphylla   |10.33008       |-73.250142      |
|unknown                |10.33521       |-73.243103      |
|unknown                |10.35922       |-73.220872      |
|Handroanthus billbergii|10.40147       |-73.191717      |
|unknown                |10.33583       |-73.241715      |
|unknown                |10.33984       |-73.237257      |
+-----------------------+---------------+----------------+
only showing top 10 rows


In [3]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import desc

feature_cols = ['decimalLongitude', 'decimalLatitude']
assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')
dataset = assembler.transform(df).select('species', 'decimalLatitude', 'decimalLongitude', 'features')

num_clusters = 10
kmeans = KMeans(k=num_clusters, seed=42, featuresCol='features', predictionCol='cluster')
model = kmeans.fit(dataset)
clustered = model.transform(dataset).cache()

print(f'Número de clusters: {num_clusters}')
print('Centroides:')
for i, center in enumerate(model.clusterCenters()):
    print(f'Cluster {i}: {center}')

print('Cantidad de puntos por cluster:')
clustered.groupBy('cluster').count().orderBy(desc('count')).show(num_clusters, truncate=False)

26/05/29 05:24:54 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Número de clusters: 10
Centroides:
Cluster 0: [-74.47810801   6.58303264]
Cluster 1: [-73.7609483   10.18146545]
Cluster 2: [-70.15211467  -3.20230287]
Cluster 3: [-74.124526     4.10827622]
Cluster 4: [-76.34266053   6.44216635]
Cluster 5: [-72.80058971  10.95576158]
Cluster 6: [-71.96227647   0.43168617]
Cluster 7: [-77.25058352   1.44862695]
Cluster 8: [-76.56751361   4.10080189]
Cluster 9: [-68.12753885   3.63161937]
Cantidad de puntos por cluster:


+-------+-----+
|cluster|count|
+-------+-----+
|1      |72004|
|5      |69871|
|4      |18292|
|8      |15797|
|0      |9266 |
|7      |7348 |
|2      |4317 |
|3      |3887 |
|6      |2066 |
|9      |1294 |
+-------+-----+



In [4]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import desc

cluster_species = clustered.groupBy('cluster', 'species').count()
window = Window.partitionBy('cluster').orderBy(desc('count'))
cluster_top = cluster_species.withColumn('rank', row_number().over(window)).filter(col('rank') <= 5)

print('Top 5 especies por cluster:')
cluster_top.orderBy('cluster', 'rank').show(50, truncate=False)

out_path = os.path.join(os.getcwd(), 'biodiversidad_clusters.parquet')
clustered.write.mode('overwrite').parquet(out_path)
print('Guardado Parquet de clusters:', out_path)

Top 5 especies por cluster:


+-------+-------------------------+-----+----+
|cluster|species                  |count|rank|
+-------+-------------------------+-----+----+
|0      |unknown                  |2230 |1   |
|0      |Leptodontium pungens     |33   |2   |
|0      |Palicourea caerulea      |27   |3   |
|0      |Tapirira guianensis      |23   |4   |
|0      |Sphagnum meridense       |22   |5   |
|1      |unknown                  |8564 |1   |
|1      |Guazuma ulmifolia        |7015 |2   |
|1      |Astronium graveolens     |5179 |3   |
|1      |Roseodendron chryseum    |3357 |4   |
|1      |Prosopis juliflora       |2761 |5   |
|2      |unknown                  |898  |1   |
|2      |Eschweilera coriacea     |55   |2   |
|2      |Virola calophylla        |43   |3   |
|2      |Rinorea lindeniana       |28   |4   |
|2      |Clathrotropis macrocarpa |25   |5   |
|3      |unknown                  |862  |1   |
|3      |Leptodontium luteum      |22   |2   |
|3      |Leptodontium wallisii    |20   |3   |
|3      |Thui

Guardado Parquet de clusters: /workspaces/bigData-Class/Notebook/biodiversidad_clusters.parquet


## Evaluación del número de clusters

Esta celda calcula el `silhouette` y el `wss` para varios valores de `k` y ayuda a elegir un número de clusters adecuado para la distribución geográfica.

In [5]:
from pyspark.ml.evaluation import ClusteringEvaluator

# Reusar el dataset con features
feature_cols = ['decimalLongitude', 'decimalLatitude']
assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')
dataset = assembler.transform(df).select('species', 'decimalLatitude', 'decimalLongitude', 'features')

evaluator = ClusteringEvaluator(featuresCol='features', predictionCol='cluster', metricName='silhouette', distanceMeasure='squaredEuclidean')
results = []
for k in range(3, 13):
    model = KMeans(k=k, seed=42, featuresCol='features', predictionCol='cluster').fit(dataset)
    predictions = model.transform(dataset)
    silhouette = evaluator.evaluate(predictions)
    wss = model.summary.trainingCost
    results.append((k, float(silhouette), float(wss)))
    print(f'k={k} -> silhouette={silhouette:.4f}, wss={wss:.2f}')

results_df = spark.createDataFrame(results, ['k', 'silhouette', 'wss'])
results_df.show()

# Mostrar la tabla de resultados para elegir el mejor k
print('Use los valores de silhouette más altos y la curva de wss para decidir k.')

ERROR:root:Exception while sending command.                         (0 + 2) / 2]
Traceback (most recent call last):
  File "/workspaces/bigData-Class/.venv/lib/python3.12/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/home/codespace/.python/current/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
ConnectionResetError: [Errno 104] Connection reset by peer

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/workspaces/bigData-Class/.venv/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/bigData-Class/.venv/lib/python3.12/site-packages/py4j/clientserver.py", line 566, in send_command
    ra

ConnectionRefusedError: [Errno 111] Connection refused

ConnectionRefusedError: [Errno 111] Connection refused

### Selección de k y análisis extendido

Con base en los resultados, elegimos un valor de `k` que equilibra un buen `silhouette` con una reducción constante de `wss` para capturar hot spots relevantes sin sobredescomponer los datos.

In [ ]:
from pyspark.sql.functions import countDistinct, count, desc, col

recommended_k = 10
print('k recomendado:', recommended_k)
print('Razon: alto silhouette con wss decreciente y un nivel de detalle adecuado para análisis de hotspots.')

if 'results_df' in globals():
    print('Resultados de evaluación de k:')
    results_df.orderBy('k').show()

# Reusar el clustering existente con k=10 y analizar características del cluster
cluster_summary = clustered.groupBy('cluster').agg(
    count('*').alias('point_count'),
    countDistinct('species').alias('species_richness')
).orderBy(desc('point_count'))

print('Resumen por cluster: cantidad de puntos y riqueza de especies')
cluster_summary.show(20, truncate=False)

cluster_species = clustered.groupBy('cluster', 'species').count()
window = Window.partitionBy('cluster').orderBy(desc('count'))
cluster_top10 = cluster_species.withColumn('rank', row_number().over(window)).filter(col('rank') <= 10)

print('Top 10 especies por cluster:')
cluster_top10.orderBy('cluster', 'rank').show(100, truncate=False)

print('Centroides del modelo final con k=10:')
for i, center in enumerate(model.clusterCenters()):
    print(f'Cluster {i}: {center}')

k recomendado: 10
Razon: alto silhouette con wss decreciente y un nivel de detalle adecuado para análisis de hotspots.
Resumen por cluster: cantidad de puntos y riqueza de especies
+-------+-----------+----------------+
|cluster|point_count|species_richness|
+-------+-----------+----------------+
|1      |72004      |1327            |
|5      |69871      |590             |
|4      |18292      |4962            |
|8      |15797      |3646            |
|0      |9266       |3043            |
|7      |7348       |2726            |
|2      |4317       |1384            |
|3      |3887       |1850            |
|6      |2066       |1031            |
|9      |1294       |772             |
+-------+-----------+----------------+

Top 10 especies por cluster:
+-------+---------------------------+-----+----+
|cluster|species                    |count|rank|
+-------+---------------------------+-----+----+
|0      |unknown                    |2230 |1   |
|0      |Leptodontium pungens       |33   |2   

In [ ]:
spark.stop()
print('Spark session cerrada.')

Spark session cerrada.
